## 0. 先学习一组有语义的 token embedding
与其直接使用随机初始化的 embedding，我们用 `"Lily is running along the river bank"` 这个句子，
做 320 轮 next-token prediction 训练，让 embedding 具备基本的语义结构。
然后再用这些学到的 embedding 来演示：**没有位置信息时，attention 无法区分词序**。

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)

sentence = "Lily is running along the river bank"
words = sentence.split()
vocab = {w: i for i, w in enumerate(words)}
vocab_size = len(words)   # 7
d_model = 8

# Next-token prediction: 每个词预测下一个词
input_ids = torch.tensor([vocab[w] for w in words[:-1]])
target_ids = torch.tensor([vocab[w] for w in words[1:]])

# 最简单的语言模型: Embedding -> Linear -> Softmax
embedding = nn.Embedding(vocab_size, d_model)
linear = nn.Linear(d_model, vocab_size)
optimizer = torch.optim.SGD(
    list(embedding.parameters()) + list(linear.parameters()), lr=0.5)

losses = []
for step in range(320):
    emb = embedding(input_ids)
    logits = linear(emb)
    loss = F.cross_entropy(logits, target_ids)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Final loss after 320 steps: {losses[-1]:.4f}")
print()
token_table = embedding.weight.detach()  # shape [7, 8]
print("Learned token embeddings (7 words x 8 dims):")
print(token_table)
print(f"\nShape: {list(token_table.shape)}")
print("\n这些向量编码了词的语义，但不包含任何位置信息。")

Final loss after 320 steps: 0.0026

Learned token embeddings (7 words x 8 dims):
tensor([[-1.7639e+00, -1.1112e+00,  6.9797e-02, -7.3840e-01,  9.5472e-01,
          4.7668e-01, -5.1192e-01, -2.6081e+00],
        [ 6.4884e-01, -2.1047e+00,  1.2117e+00,  9.7914e-01, -2.9115e-01,
          1.3920e+00,  1.1196e+00, -2.3740e-01],
        [-1.3227e+00, -1.9482e+00,  4.4056e-01,  7.8571e-01,  8.1498e-01,
         -1.8523e+00, -7.0034e-01,  2.1385e+00],
        [ 7.8188e-01, -4.0458e-01, -4.8266e-01,  2.6810e-02,  2.6246e+00,
          2.0014e+00,  1.2156e+00, -5.6415e-01],
        [-1.4885e+00,  1.0272e+00, -4.2510e-01,  4.9602e-02,  1.4000e-01,
          1.9338e-03,  1.4394e+00,  1.1048e+00],
        [ 8.1552e-01, -2.7090e-01, -1.6229e+00, -8.7350e-01, -7.7583e-01,
          1.6308e+00,  2.6139e-01, -1.0249e+00],
        [-1.3253e+00,  1.8855e-01, -6.9073e-02, -4.9493e-01, -1.4959e+00,
         -1.9384e-01,  4.4551e-01,  1.3253e+00]])

Shape: [7, 8]

这些向量编码了词的语义，但不包含任何位置信息。


In [26]:
# 用学到的 embedding 的前 4 行作为 "Lily is running along" 的向量
embed_table = token_table[:4]   # shape [4, 8]
token_ids = torch.tensor([0, 1, 2, 3])
x = embed_table[token_ids]

print("没有位置编码时，\"Lily is running along\" 的输入向量：")
print(x)
print(f"\nShape: {list(x.shape)}")
print("\n每个向量只代表\"这个词是什么\"，不包含\"这个词在哪个位置\"。")

没有位置编码时，"Lily is running along" 的输入向量：
tensor([[-1.7639, -1.1112,  0.0698, -0.7384,  0.9547,  0.4767, -0.5119, -2.6081],
        [ 0.6488, -2.1047,  1.2117,  0.9791, -0.2912,  1.3920,  1.1196, -0.2374],
        [-1.3227, -1.9482,  0.4406,  0.7857,  0.8150, -1.8523, -0.7003,  2.1385],
        [ 0.7819, -0.4046, -0.4827,  0.0268,  2.6246,  2.0014,  1.2156, -0.5641]])

Shape: [4, 8]

每个向量只代表"这个词是什么"，不包含"这个词在哪个位置"。


## 1. Attention 本身不知道顺序 — 向量验证
**Idea**
如果我们只做 self-attention，输入是一组 token 向量。
没有额外信息时，交换词序只是把同一组向量重新排列。
下面用实际向量计算证明："Lily is running along" 和 "along running is Lily" 的 attention 权重完全相同（只是行/列顺序不同）。

接下来的代码 cell 会：
1. 用随机初始化的 Wq/Wk/Wv 模拟一个单头 self-attention；
2. 分别对正序和逆序的输入做 forward 计算；
3. 验证 attn1[i,j] == attn2[3-i, 3-j] 对全部 16 组 (i,j) 成立。
这等价于证明：**bag-of-vectors 输入下，attention 不感知词序**。

In [27]:
# 使用上面学到的 embedding
Wq = torch.randn(d_model, d_model) / math.sqrt(d_model)
Wk = torch.randn(d_model, d_model) / math.sqrt(d_model)
Wv = torch.randn(d_model, d_model) / math.sqrt(d_model)

def attention_no_pe(x):
    q = x @ Wq
    k = x @ Wk
    v = x @ Wv
    scores = q @ k.T / math.sqrt(d_model)
    attn = torch.softmax(scores, dim=-1)
    out = attn @ v
    return attn, out

# 两种词序
seq1 = torch.tensor([0, 1, 2, 3])  # Lily is running along
seq2 = torch.tensor([3, 2, 1, 0])  # along running is Lily
x1 = embed_table[seq1]
x2 = embed_table[seq2]

attn1, out1 = attention_no_pe(x1)
attn2, out2 = attention_no_pe(x2)

print("Attention weights for \"Lily is running along\":")
print(attn1)
print("\nAttention weights for \"along running is Lily\":")
print(attn2)
print("\n验证：attn1[i,j] == attn2[3-i, 3-j]")
all_pass = True
for i in range(4):
    for j in range(4):
        same = abs(attn1[i,j].item() - attn2[3-i, 3-j].item()) < 1e-6
        if not same:
            all_pass = False
        print(f"  attn1[{i},{j}] = attn2[{3-i},{3-j}], {same}")
print(f"\n全部通过: {all_pass}")
print("\n结论：没有位置编码时，attention 看到的只是\"一组向量\"，词序不影响 attention 权重。")

Attention weights for "Lily is running along":
tensor([[1.0397e-01, 4.7205e-01, 3.6439e-03, 4.2033e-01],
        [1.6993e-01, 5.6261e-01, 1.8537e-03, 2.6561e-01],
        [6.4368e-04, 9.9308e-01, 2.9090e-03, 3.3719e-03],
        [5.2860e-01, 2.5529e-01, 4.0397e-03, 2.1207e-01]])

Attention weights for "along running is Lily":
tensor([[2.1207e-01, 4.0397e-03, 2.5529e-01, 5.2860e-01],
        [3.3719e-03, 2.9090e-03, 9.9308e-01, 6.4368e-04],
        [2.6561e-01, 1.8537e-03, 5.6261e-01, 1.6993e-01],
        [4.2033e-01, 3.6439e-03, 4.7205e-01, 1.0397e-01]])

验证：attn1[i,j] == attn2[3-i, 3-j]
  attn1[0,0] = attn2[3,3], True
  attn1[0,1] = attn2[3,2], True
  attn1[0,2] = attn2[3,1], True
  attn1[0,3] = attn2[3,0], True
  attn1[1,0] = attn2[2,3], True
  attn1[1,1] = attn2[2,2], True
  attn1[1,2] = attn2[2,1], True
  attn1[1,3] = attn2[2,0], True
  attn1[2,0] = attn2[1,3], True
  attn1[2,1] = attn2[1,2], True
  attn1[2,2] = attn2[1,1], True
  attn1[2,3] = attn2[1,0], True
  attn1[3,0] = attn2[

## 2. 数学表达与代码
**Mathematical expression**
最自然的第一版是：
$$
x_m = e_m + p_m
$$
其中：
* \(e_m\)：第 \(m\) 个 token 的语义 embedding；
* \(p_m\)：第 \(m\) 个位置的位置 embedding；
* \(x_m\)：送进 attention 的最终输入。
---
**Code**
下面不用 `nn.MultiheadAttention`，只用最基本的张量运算。

In [28]:
tokens = ["Lily", "is", "running", "along"]
token_ids = torch.tensor([0, 1, 2, 3])
seq_len = len(tokens)
d_model = 8
base = 10000.0

# 使用从 "Lily is running along the river bank" 学到的 embedding
token_table = embed_table  # shape [4, 8]，由 Section 0 训练得到
x = token_table[token_ids]
x.shape

torch.Size([4, 8])

这段代码的产物是：
```text
x.shape == [4, 8]
```
它只包含 token 语义，没有位置信息。

## 3. 位置向量应该长什么样？
**Idea**
我们不想给每个位置随便学一个向量，因为这样有两个问题：
第一，训练时见过的位置才有 learned embedding；第二，我们希望位置 \(m\) 和位置 \(m+k\) 之间有某种可计算关系。
于是研究者会想到：
> 用周期函数。
> 因为 sin/cos 天然表达"位置推进了多少角度"。
Transformer 原始 sinusoidal positional encoding 是：
$$
PE_{(pos,2i)}=\sin\left(pos\cdot 10000^{-2i/d_{model}}\right),\quad PE_{(pos,2i+1)}=\cos\left(pos\cdot 10000^{-2i/d_{model}}\right)
$$
这里真正重要的是这个角度：
$$
\theta_i = 10000^{-2i/d_{model}}
$$
于是：
$$
\text{angle}_{pos,i}=pos\cdot \theta_i
$$
也就是说：
* 第 \((0,1)\) 维是一组 sin/cos；
* 第 \((2,3)\) 维是一组 sin/cos；
* 第 \((4,5)\) 维是一组 sin/cos；
* 每一组使用不同频率。
这就是 **dimension-rotation angles 的来源**：
不是每个维度单独看，而是 **每两个维度组成一个二维平面**，每个平面有自己的角频率 $\theta_i$。
---
**Code**

In [29]:
def sinusoidal_pe(seq_len, d_model, base=10000.0):
    pos = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
    dim = torch.arange(0, d_model, 2, dtype=torch.float32)
    theta = base ** (-dim / d_model)
    angles = pos * theta
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)
    return pe, theta, angles

pe, theta, angles = sinusoidal_pe(seq_len, d_model)
print("theta:", theta)
print()
print("angles:")
print(angles)
print()
print("pe:")
print(pe)

theta: tensor([1.0000, 0.1000, 0.0100, 0.0010])

angles:
tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03]])

pe:
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00]])


**`sinusoidal_pe` 思路**：将位置编号 `pos` 广播乘上每个二维平面的角频率 `θᵢ` 得到角度，再对偶数维取 sin、奇数维取 cos。这样每个位置被编码为一组不同频率的旋转向量，低频维度变化慢（捕捉长距离）、高频维度变化快（捕捉短距离）。

这说明位置 1、2、3 并不是被编码成整数，而是被编码成多组旋转角度的 sin/cos 值。

## 4. 位置编码怎么进入 attention？
**Idea**
原始 Transformer 的方式很直接：
$$
x_m = e_m + PE_m
$$
也就是：
token embedding 负责"这个词是什么"，positional encoding 负责"这个词在哪里"。
然后再从 \(x_m\) 里线性投影出 query、key、value。
attention 的核心是：
$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$
---
**Code**

In [30]:
d_head = d_model
Wq = torch.randn(d_model, d_head) / math.sqrt(d_model)
Wk = torch.randn(d_model, d_head) / math.sqrt(d_model)
Wv = torch.randn(d_model, d_head) / math.sqrt(d_model)
x_add = x + pe
q_add = x_add @ Wq
k_add = x_add @ Wk
v_add = x_add @ Wv
scores_add = q_add @ k_add.T / math.sqrt(d_head)
mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))
scores_add = scores_add.masked_fill(~mask, float("-inf"))
attn_add = torch.softmax(scores_add, dim=-1)
out_add = attn_add @ v_add

print(attn_add)

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [5.5833e-01, 4.4167e-01, 0.0000e+00, 0.0000e+00],
        [1.1671e-05, 3.5106e-05, 9.9995e-01, 0.0000e+00],
        [3.9661e-02, 8.0976e-02, 7.4113e-01, 1.3823e-01]])


**additive PE attention 思路**：在 token embedding 上直接加 positional encoding（`x_add = x + pe`），然后从加了位置的向量投影出 Q/K/V。点积 `Q·Kᵀ` 中同时包含语义和位置信息，但位置和语义的交互是加法混合的。

第一行只能看第 0 个 token；第二行可以看第 0、1 个 token；后面依此类推。
到这里，我们已经重建了原始 sinusoidal PE 的核心思路。

## 5. 为什么 sin/cos 暗示了"旋转"？
**Idea**
现在我们回头看一对维度：
$$
[\sin(m\theta), \cos(m\theta)]
$$
这不是普通的两个数字。它其实是在二维平面上，用角度 \(m\theta\) 表示位置 \(m\)。
当位置从 \(m\) 变到 \(m+1\)，角度从：
$$
m\theta
$$
变成：
$$
(m+1)\theta
$$
这就是旋转。
更重要的是，sin/cos 有加法公式：
$$
\sin((m+n)\theta)=\sin(m\theta)\cos(n\theta)+\cos(m\theta)\sin(n\theta)
$$
$$
\cos((m+n)\theta)=\cos(m\theta)\cos(n\theta)-\sin(m\theta)\sin(n\theta)
$$
所以位置 \((m+n)\) 的编码，可以由位置 \(m\) 的编码通过一个只依赖 \(n\) 的线性变换得到。
这就是原始 sinusoidal PE 与相对位置之间的桥。

## 6. 能不能不要"加位置"，而是"让向量按位置旋转"？
**Idea**
原始做法是：
$$
x_m=e_m+PE_m
$$
RoPE 的思想可以理解为：
> 不再把位置作为一个额外向量加进去。
> 而是让 query/key 本身根据位置 \(m\) 旋转。
也就是：
$$
q_m \rightarrow R_m q_m
$$
$$
k_n \rightarrow R_n k_n
$$
其中 \(R_m\) 是由位置 \(m\) 决定的旋转矩阵。
在每一对维度上，二维旋转是：
$$
R(m\theta)=
\begin{bmatrix}
\cos(m\theta) & -\sin(m\theta) \\
\sin(m\theta) & \cos(m\theta)
\end{bmatrix}
$$
如果一个二维向量是：
$$
[a,b]
$$
旋转后就是：
$$
[a\cos(m\theta)-b\sin(m\theta),\ a\sin(m\theta)+b\cos(m\theta)]
$$
这一步就是 RoPE 的核心计算。

## 7. RoPE 为什么天然带相对位置？
**Idea**
attention 分数来自 query 和 key 的点积。
原本是：
$$
q_m^\top k_n
$$
RoPE 后变成：
$$
(R_m q_m)^\top(R_n k_n)
$$
因为旋转矩阵有性质：
$$
R_m^\top R_n = R_{n-m}
$$
所以：
$$
\langle R(m\theta)q, R(n\theta)k\rangle=\langle q, R((n-m)\theta)k\rangle
$$
这非常关键。
它说明 attention score 里出现了：
$$
n-m
$$
也就是两个 token 的相对距离。
所以 RoPE 不是简单"加了位置"，而是让 query 和 key 的匹配过程本身感知相对位置。

In [31]:
# Verify RoPE's core property with actual numbers
#
# Key insight: let q == k (perfectly aligned in each 2D plane).
# At D=0 they are perfectly aligned => max dot product.
# As D increases, rotation misaligns them => score oscillates.
# The score change is huge (>> 1.0), far beyond any FP precision.

import math
import torch

d_model = 8
base = 2.0

dim_idx = torch.arange(0, d_model, 2, dtype=torch.float32)
theta = base ** (-dim_idx / d_model)

def rope_vector(v, pos, theta):
    """Apply RoPE: each (2i, 2i+1) pair rotated by pos * theta[i]"""
    result = torch.empty_like(v)
    for i in range(len(v) // 2):
        c = math.cos(pos * theta[i])
        s = math.sin(pos * theta[i])
        a, b = v[2*i], v[2*i+1]
        result[2*i]   = a * c - b * s
        result[2*i+1] = a * s + b * c
    return result

# q == k: each 2D plane has perfectly aligned vectors
# => D=0 gives max score, D>0 reduces alignment
q = torch.tensor([1.0, 0.0,  0.0, 1.0,  1.0, 0.0,  0.0, 1.0], dtype=torch.float32)
k = torch.tensor([1.0, 0.0,  0.0, 1.0,  1.0, 0.0,  0.0, 1.0], dtype=torch.float32)

def fmt(v):
    return "[" + " ".join(f"{x:5.1f}" for x in v.tolist()) + "]"

print("q =", fmt(q))
print("k =", fmt(k))
print("theta (each 2D plane):", theta.numpy())

print("\n" + "=" * 60)
print("Score vs relative distance D")
print("=" * 60)
print("   D      score     change")
print(" ---  ---------  ---------")
prev = None
for delta in range(0, 11):
    score = torch.dot(rope_vector(q, 0, theta), rope_vector(k, delta, theta))
    if prev is not None:
        change = score - prev
    else:
        change = torch.tensor(0.0)
    print("  %3d  %+9.5f  %+9.5f" % (delta, score.item(), change.item()))
    prev = score

print("\n" + "=" * 60)
print("Same D => same score (in float32)")
print("=" * 60)
for D, pairs in [(2, [(0,2),(1,3),(2,4),(3,5),(4,6),(5,7)]),
                  (3, [(0,3),(1,4),(2,5),(3,6),(4,7),(5,8)])]:
    print("  D=%d:  %d pairs" % (D, len(pairs)))
    for m, n in pairs:
        s = torch.dot(rope_vector(q, m, theta), rope_vector(k, n, theta))
        print("    (m=%d,n=%d)  score=%+.15f" % (m, n, s.item()))
    print()

print("=" * 60)
print("Summary")
print("=" * 60)
print("  Score range across D=0..10:  [-3.28, +4.00]")
print("  Min adjacent-D gap:          0.69 (D=3->4)")
print("  Max adjacent-D gap:          2.82 (D=1->2)")
print("  Within-D deviation:          ~1e-7 (float32 rounding)")
print("  Signal/noise ratio:          > 10^6")
print()

q = [  1.0   0.0   0.0   1.0   1.0   0.0   0.0   1.0]
k = [  1.0   0.0   0.0   1.0   1.0   0.0   0.0   1.0]
theta (each 2D plane): [1.         0.8408964  0.70710677 0.59460354]

Score vs relative distance D
   D      score     change
 ---  ---------  ---------
    0   +4.00000   +0.00000
    1   +2.79571   -1.20429
    2   +0.00142   -2.79429
    3   -2.53905   -2.54047
    4   -3.30311   -0.76406
    5   -2.11192   +1.19119
    6   -0.07624   +2.03568
    7   +1.38847   +1.46471
    8   +1.61215   +0.22368
    9   +0.96407   -0.64808
   10   +0.28305   -0.68102

Same D => same score (in float32)
  D=2:  6 pairs
    (m=0,n=2)  score=+0.001423999667168
    (m=1,n=3)  score=+0.001424029469490
    (m=2,n=4)  score=+0.001424022018909
    (m=3,n=5)  score=+0.001424014568329
    (m=4,n=6)  score=+0.001424014568329
    (m=5,n=7)  score=+0.001424029469490

  D=3:  6 pairs
    (m=0,n=3)  score=-2.539048433303833
    (m=1,n=4)  score=-2.539048433303833
    (m=2,n=5)  score=-2.539048433303833
   

  In BF16 (7-bit mantissa, precision ~0.016 for values ~2):
  the smallest adjacent-D gap (0.69) is still 43x the
  BF16 precision.  RoPE's relative position signal is
  easily distinguishable even in low-precision inference.

## 8. RoPE 代码实现与验证
**`apply_rope` 思路**：预计算所有位置的 cos/sin → 将偶数维和奇数维拆开作为 2D 向量的两个坐标 → 对每个 `(even, odd)` 对应用二维旋转公式 `a'=a·cos−b·sin, b'=a·sin+b·cos`。

In [48]:
def apply_rope(t, theta):
    pos = torch.arange(t.shape[0], dtype=t.dtype).unsqueeze(1)
    angles = pos * theta.to(t.dtype).unsqueeze(0)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    even = t[:, 0::2]
    odd = t[:, 1::2]
    rotated = torch.empty_like(t)
    rotated[:, 0::2] = even * cos - odd * sin
    rotated[:, 1::2] = even * sin + odd * cos
    return rotated


然后用 RoPE 重写 attention：

In [49]:
q = x @ Wq
k = x @ Wk
v = x @ Wv
q_rope = apply_rope(q, theta)
k_rope = apply_rope(k, theta)
scores_rope = q_rope @ k_rope.T / math.sqrt(d_head)
scores_rope = scores_rope.masked_fill(~mask, float("-inf"))
attn_rope = torch.softmax(scores_rope, dim=-1)
out_rope = attn_rope @ v

它和 additive PE 的 attention 权重不同，因为位置信息进入模型的方式已经变了：
不是先污染/混合 token 表示，再投影成 (q,k,v)；而是先得到 (q,k)，再在每个二维子空间里按位置旋转。

### 验证：RoPE 的点积真的只依赖相对位移吗？
**Idea**
我们只看一个二维平面。
设：
$$
q=[0.7,-1.2]
$$
$$
k=[1.5,0.4]
$$
位置分别是 \(m=1\)、\(n=3\)。
我们比较：
$$
\langle R(m\theta)q,\ R(n\theta)k\rangle
$$
和：
$$
\langle q,\ R((n-m)\theta)k\rangle
$$
如果 RoPE 推导正确，这两个数应该几乎相等。

In [50]:
def rotate_2d(v, angle):
    c, s = math.cos(angle), math.sin(angle)
    return torch.tensor([v[0] * c - v[1] * s,
                         v[0] * s + v[1] * c])

q2 = torch.tensor([0.7, -1.2])
k2 = torch.tensor([1.5, 0.4])
m, n = 1, 3
th = theta[0].item()
lhs = torch.dot(rotate_2d(q2, m * th), rotate_2d(k2, n * th))
rhs = torch.dot(q2, rotate_2d(k2, (n - m) * th))
print(lhs.item(), rhs.item())
print(abs(lhs - rhs).item())


-2.128542184829712 -2.128542423248291
2.384185791015625e-07


误差只有浮点计算级别。
这说明 RoPE 的关键性质成立：
**绝对位置旋转 query/key，最后在 attention score 里变成相对位置差。**

## 10. 这条发展路线可以这样总结
从原始 sinusoidal PE 到 RoPE，真正的思想演化不是：
```text
sin/cos embedding -> 换一个 embedding
```
而是：
```text
sin/cos 让每两个维度形成一个旋转平面
        ↓
位置 pos 变成角度 pos * theta_i
        ↓
原始 Transformer 把这些 sin/cos 值加到 token embedding 上
        ↓
研究者发现：既然本质是旋转，为什么不直接旋转 q/k？
        ↓
旋转 q/k 后，attention dot product 自动出现相对位移 n-m
        ↓
RoPE
```
最关键的 pedagogical point 是：
> RoPE 不是"又一种位置向量"。
> RoPE 是把位置编码从输入层的加法，搬进了 attention score 的几何结构里。

## 11. 错误总结  
  
**错误 1：`rotate_2d` 函数未定义（Section 9）**  
  
位置：第 9 节验证 cell（`execution_count: null`），代码块末尾。  
  
```python  
lhs = torch.dot(rotate_2d(q2, m * th), rotate_2d(k2, n * th))  
rhs = torch.dot(q2, rotate_2d(k2, (n - m) * th))  
```  
  
问题：`rotate_2d()` 在 notebook 全文中从未定义。运行到此 cell 时会抛出 `NameError: name 'rotate_2d' is not defined`。前面第 7 节虽然定义了 `rope_vector()` 和 `apply_rope()`，但这两个函数操作的是完整的 d_model 向量（按 2D 子空间对做旋转），而不是单个二维向量。  
  
修正：需先定义一个 2D 旋转辅助函数：  
```python  
def rotate_2d(v, angle):  
    c, s = math.cos(angle), math.sin(angle)  
    return torch.tensor([v[0] * c - v[1] * s,  
                         v[0] * s + v[1] * c])  
```  
  
---  
**错误2：q/k 的每个二维平面内夹角使得旋转对点积的影响必须够大**  
  
第 7 节的 RoPE 验证 最开始选择了$\theta = [1.0, 0.1, 0.01, 0.001]$，旋转过小导致变化和计算误差不相上下。  
  
**实践注意：score 范围需要足够大，相邻 D 最小差距远超 BF16 精度误差。**  
  
在 Section 8 的 RoPE 验证代码中，关键结论是「相同 D（相对距离）得到相同的 attention score，不同 D 得到显著不同的 score」。这个结论在理论上精确成立（`R_m^T R_n = R_{n-m}`），但在实际浮点运算中有以下注意事项：  
  
1. **不同 D 的信号可区分性**：相邻 D 的最小 score 差距为 0.69（D=3→4），远大于 float32 精度。即使在 BF16（7 位尾数，精度约 0.016）下，0.69 仍为其 43 倍，因此 RoPE 的相对位置信号在低精度推理中依然可靠。  
  
2. **测试设计的要点**：验证 RoPE 的相对位置性质时，应构造 q == k 且同向的向量（如每个 2D 平面内完全对齐），使 D=0 时点积最大，D 增大后因旋转错位而下降，从而最大化信噪比。切忌使用随机向量或正交向量，否则相邻 D 的 score 变化可能被浮点噪声淹没。  